In [1]:
import re
import json
import torch
import sys
from pathlib import Path
import numpy as np

from tqdm.auto import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
from donut import JSONParseEvaluator

The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-240807/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# in jupyter (lab / notebook), based on notebook path
module_path = str(Path.cwd().parents[0])

# # in standard python
# module_path = str(Path.cwd(__file__).parents[0] / "py")

if module_path not in sys.path:
    sys.path.append(module_path)

In [3]:
from data.custom_dataset import CustomImageDataset

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [5]:
torch.cuda.empty_cache()

In [6]:
!nvidia-smi

Mon Dec  2 11:54:33 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM-64GB            On | 00000000:1D:00.0 Off |                    0 |
| N/A   43C    P0               61W / 466W|      3MiB / 65536MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [7]:
image_path = "/leonardo_work/IscrC_HeR-T/weiwei/data/20240517/HeR-T_data/img_600x800/test"
dataset = CustomImageDataset(image_path)

In [8]:
processor = DonutProcessor.from_pretrained("/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results/600x800/processor")
model = VisionEncoderDecoderModel.from_pretrained("/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results/600x800/model")

In [9]:
model.eval()
model.to(device)

VisionEncoderDecoderModel(
  (encoder): DonutSwinModel(
    (embeddings): DonutSwinEmbeddings(
      (patch_embeddings): DonutSwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DonutSwinEncoder(
      (layers): ModuleList(
        (0): DonutSwinStage(
          (blocks): ModuleList(
            (0-1): 2 x DonutSwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): DonutSwinAttention(
                (self): DonutSwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
           

In [9]:
# api key so that it doesn't ask me for it
# wandb.init(mode="offline")
# wandb.login(key=confidential.api_key)
# wandb_logger = WandbLogger(project="HeR-T-trial-evaluation-241120", name="scratch_evaluation_6epoch")

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


In [10]:
output_list, accs, files = [], [], []

for idx, sample in tqdm(enumerate(dataset), total=len(dataset)):
    # Do some execption handling for wierd files
    try:
        # Load the image
        image = sample[0].convert("RGB")
        
        # Check if the image is truncated
        image.load()
    except OSError as e:
        if "image file is truncated" in str(e):
            print(f"Warning: Skipping truncated image")
            continue
        else:
            raise
                
    # prepare encoder inputs
    pixel_values = processor(sample[0].convert("RGB"), return_tensors="pt").pixel_values
    pixel_values = pixel_values.to(device)
    # prepare decoder inputs
    task_prompt = "<s_herbarium>"
    decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids
    decoder_input_ids = decoder_input_ids.to(device)
        
    # autoregressively generate sequence
    outputs = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=model.decoder.config.max_position_embeddings,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=1,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )

    # turn into JSON
    seq = processor.batch_decode(outputs.sequences)[0]
    seq = seq.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
    seq = re.sub(r"<.*?>", "", seq, count=1).strip()  # remove first task start token
    seq = processor.token2json(seq)
    
    ground_truth = json.loads(sample[1])
    evaluator = JSONParseEvaluator()
    score = evaluator.cal_acc(seq, ground_truth)
    file = sample[2]

    accs.append(score)
    output_list.append(seq)
    files.append(file)

100%|████████████████████████████████████████████████| 6893/6893 [33:57<00:00,  3.38it/s]


In [ ]:
# wandb.log({"predictions": table})

In [11]:
scores = {"accuracies": accs, "mean_accuracy": np.mean(accs)}
print(f"Accuracy list length : {len(accs)}")
print("Mean_accuracy: ", np.mean(accs))
print("Median accuracy:", np.median(accs))

Accuracy list length : 6893
Mean_accuracy:  0.5861875643757822
Median accuracy: 0.6


In [12]:
with open("/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results/600x800/output_pre.txt", 'w') as output_pre:
    for row in output_list:
        output_pre.write(str(row) + '\n')

In [13]:
with open("/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results/600x800/acc_pre.txt", 'w') as acc_pre:
    for row in accs:
        acc_pre.write(str(row) + '\n')

In [14]:
with open("/leonardo_work/IscrC_HeR-T/weiwei/HeR-T-Retraining/results/600x800/file_names.txt", 'w') as file_names:
    for row in files:
        file_names.write(str(row) + '\n')

In [15]:
# wandb.finish()

In [16]:
# mean_without_worst = np.mean(np.sort(accs)[883:])
# print("Mean accuracy (excluding worst cases):", mean_without_worst)

Mean accuracy (excluding worst cases): 0.8185248932040329


In [ ]:
# # get indices of worst 10 predictions
# worst_idxs = np.argsort(accs)[:100].tolist()

# # prepare decoder inputs
# task_prompt = "<s_herbarium>"
# decoder_input_ids = processor.tokenizer(task_prompt, 
#                                         add_special_tokens=False, 
#                                         return_tensors="pt").input_ids
# decoder_input_ids = decoder_input_ids.to(device)

# for idx in worst_idxs:
#     sample = dataset[idx]

#     # prepare encoder inputs
#     pixel_values = processor(sample["image"].convert("RGB"), 
#                              return_tensors="pt").pixel_values
#     pixel_values = pixel_values.to(device)

#     # autoregressively generate sequence
#     outputs = model.generate(
#         pixel_values,
#         decoder_input_ids=decoder_input_ids,
#         max_length=model.decoder.config.max_position_embeddings,
#         early_stopping=True,
#         pad_token_id=processor.tokenizer.pad_token_id,
#         eos_token_id=processor.tokenizer.eos_token_id,
#         use_cache=True,
#         num_beams=1,
#         bad_words_ids=[[processor.tokenizer.unk_token_id]],
#         return_dict_in_generate=True,
#     )

#     # turn into JSON
#     seq = processor.batch_decode(outputs.sequences)[0]
#     seq = seq.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
#     seq = re.sub(r"<.*?>", "", seq, count=1).strip()  # remove first task start token
#     seq = processor.token2json(seq)
    
#     print(f"Ground Truth: {sample['ground_truth']}\n")
#     print(f"Prediction: {seq}\n")
#     print(f"Score: {accs[idx]}\n")
#     display(sample["image"])